# TrendPulse — Part 2: NumPy & Pandas Data Wrangling

**Project:** What's Actually Trending Right Now  
**Part marks:** 25  
**Countries:** US, GB, IN  
**Data source:** Trending YouTube Video Statistics dataset supplied for this assignment.

## How to run
1. Keep this notebook in `part2_numpy_pandas/`.
2. Keep the four CSV files in the sibling `data/` folder: `USvideos.csv`, `GBvideos.csv`, `INvideos.csv`, `categories.csv`.
3. Run every cell from top to bottom (`Runtime → Restart and run all` in Google Colab).
4. This notebook uses only local CSV files. It makes no API calls and does not scrape the web.

**Dependencies:** Python 3.10+; `numpy`, `pandas`, `matplotlib` is not required. Install with `pip install -r requirements.txt`.

**Design note:** Raw rows represent a video appearing on one country's trending list on one date. For this analysis, a duplicate is the same `video_id` + `trending_date` + `country`. Repeated appearances on different dates are not duplicates.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
COUNTRY_FILES = {
    "US": DATA_DIR / "USvideos.csv",
    "GB": DATA_DIR / "GBvideos.csv",
    "IN": DATA_DIR / "INvideos.csv",
}
CATEGORY_FILE = DATA_DIR / "categories.csv"

missing_files = [str(p) for p in [*COUNTRY_FILES.values(), CATEGORY_FILE] if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Required CSV file(s) not found: " + ", ".join(missing_files) +
        ". Put the data/ folder beside this notebook."
    )

print("All four local CSV files were found.")


All four local CSV files were found.


## 1. Load and inspect the raw country files

The country DataFrames are loaded independently, tagged with a `country` column, and then concatenated. `trending_date` uses the dataset's `YY.DD.MM` representation; `publish_time` is parsed as a UTC timestamp.


In [2]:
country_dfs = {}
for country, file_path in COUNTRY_FILES.items():
    country_df = pd.read_csv(file_path)
    country_df["country"] = country
    country_dfs[country] = country_df

for country, country_df in country_dfs.items():
    print(f"\\n===== {country} =====")
    print("shape:", country_df.shape)
    display(country_df.head())
    print("\\ninfo:")
    country_df.info()
    print("\\ndescribe:")
    display(country_df.describe(include="all").T)

combined_raw = pd.concat(country_dfs.values(), ignore_index=True)
print("\\nCombined raw shape:", combined_raw.shape)
display(combined_raw.head())
combined_raw.info()
display(combined_raw.describe(include="all").T)


\n===== US =====
shape: (16080, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27T22:21:47.000Z,Festival|Guitar,8083,176.0,8,19.0,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,UltimateVlog96,27,2017-11-09T13:20:20.000Z,Festival,373161,11305.0,167,891.0,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US


\ninfo:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080 entries, 0 to 16079
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                16080 non-null  object 
 1   trending_date           16080 non-null  object 
 2   title                   16080 non-null  object 
 3   channel_title           16080 non-null  object 
 4   category_id             16080 non-null  int64  
 5   publish_time            16080 non-null  object 
 6   tags                    16080 non-null  object 
 7   views                   16080 non-null  int64  
 8   likes                   16076 non-null  float64
 9   dislikes                16080 non-null  int64  
 10  comment_count           16068 non-null  float64
 11  thumbnail_link          16080 non-null  object 
 12  comments_disabled       16080 non-null  bool   
 13  ratings_disabled        16080 non-null  bool   
 14  video_error_or_removed  16080 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
video_id,16080,5068,BsTMdCTYs7P,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,16080,80,17.30.12,205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,16080,4403,Sunny Park - Bangalore (Live),33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,16080,408,QuietChallenge75,644,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,16080.0,NaN,NaN,NaN,18.975871,7.590729,1.0,10.0,23.0,24.0,28.0
publish_time,16080,5064,2017-11-11T12:46:59.000Z,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags,16080,2501,[none],1164,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,16080.0,NaN,NaN,NaN,1012289.201741,3746760.856871,1621.0,117307.75,289072.5,750719.0,129229245.0
likes,16076.0,NaN,NaN,NaN,39504.637534,171941.394484,0.0,3353.0,9229.0,26267.0,6795138.0
dislikes,16080.0,NaN,NaN,NaN,2115.772015,7973.92775,0.0,194.0,540.0,1506.0,247546.0


\n===== GB =====
shape: (15075, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,u_nKn025TZS,17.14.11,Wild Winter Prank,OfficialRecipe78,28,2017-10-18T07:42:02.000Z,[none],84344,2155.0,163,475.0,https://i.ytimg.com/vi/u_nKn025TZS/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
1,FoQAEGxMMQD,17.14.11,Wild Cricket Tutorial FINALE,FinalTrailer97,26,2017-11-14T12:56:18.000Z,Minecraft|Paris,230434,5412.0,330,1704.0,https://i.ytimg.com/vi/FoQAEGxMMQD/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
2,wHqDBuStXq3,17.14.11,Epic Space Challenge,UltimateReview54,23,2017-11-13T06:19:10.000Z,Paris,338852,16813.0,1333,1385.0,https://i.ytimg.com/vi/wHqDBuStXq3/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
3,ZId0IGzSaaZ,17.14.11,Quiet Paris Prank LIVE?!,FinalUpdate79,24,2017-11-14T11:37:33.000Z,Phones,160896,4221.0,483,1342.0,https://i.ytimg.com/vi/ZId0IGzSaaZ/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
4,1-xA4eRrYSH,17.14.11,Nova Kane - Chess (Official Video)!,QuietInterview9,10,2017-11-14T03:13:26.000Z,Chess|Minecraft|Festival|Guitar,59441,1475.0,77,60.0,https://i.ytimg.com/vi/1-xA4eRrYSH/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB


... (6 outputs left out)

\nCombined raw shape: (46230, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27T22:21:47.000Z,Festival|Guitar,8083,176.0,8,19.0,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,UltimateVlog96,27,2017-11-09T13:20:20.000Z,Festival,373161,11305.0,167,891.0,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46230 entries, 0 to 46229
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                46230 non-null  object 
 1   trending_date           46230 non-null  object 
 2   title                   46230 non-null  object 
 3   channel_title           46230 non-null  object 
 4   category_id             46230 non-null  int64  
 5   publish_time            46230 non-null  object 
 6   tags                    46230 non-null  object 
 7   views                   46230 non-null  int64  
 8   likes                   46218 non-null  float64
 9   dislikes                46230 non-null  int64  
 10  comment_count           46194 non-null  float64
 11  thumbnail_link          46230 non-null  object 
 12  comments_disabled       46230 non-null  bool   
 13  ratings_disabled        46230 non-null  bool   
 14  video_error_or_removed  46230 non-null

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
video_id,46230,14699,iAvRHGeUpBR,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,46230,80,17.30.12,608,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,46230,10404,Sunny Park - Bangalore (Live),66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,46230,438,QuietChallenge75,2464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,46230.0,NaN,NaN,NaN,18.645339,7.646745,1.0,10.0,22.0,24.0,28.0
publish_time,46230,14678,2017-11-27T11:50:56.000Z,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags,46230,6056,[none],3785,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,46230.0,NaN,NaN,NaN,1178603.180748,5916188.240094,1066.0,129463.25,318277.5,824050.5,360760164.0
likes,46218.0,NaN,NaN,NaN,45800.054849,296955.393069,0.0,3656.0,10201.0,28987.75,27537822.0
dislikes,46230.0,NaN,NaN,NaN,2325.983625,10518.962756,0.0,213.0,594.0,1666.0,618175.0


## 2. Data-quality audit and cleaning

The audit reports every column's missing count and percentage. Missing `likes` and `comment_count` are filled with zero so the numerical calculations remain defined; missing `description` is filled with an empty string because description text is not used in the metrics. Rows missing essential identifiers or dates are dropped. Duplicate keys are removed, while a video on a different trending date remains a valid separate observation.

Rows marked `video_error_or_removed=True` are reported and excluded from engagement analysis because they represent removed/errored videos. The original raw count is retained for the audit.


In [3]:
# Parse dates and standardize numerical / Boolean columns.
combined = combined_raw.copy()

combined["trending_date"] = pd.to_datetime(
    combined["trending_date"].astype("string"),
    format="%y.%d.%m",
    errors="coerce"
)
combined["publish_time"] = pd.to_datetime(
    combined["publish_time"], errors="coerce", utc=True
)

numeric_cols = ["category_id", "views", "likes", "dislikes", "comment_count"]
for col in numeric_cols:
    combined[col] = pd.to_numeric(combined[col], errors="coerce")

boolean_cols = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]
for col in boolean_cols:
    combined[col] = combined[col].astype("boolean")

# Report missing values before filling or dropping anything.
missing_audit = pd.DataFrame({
    "missing_count": combined.isna().sum(),
    "missing_percent": (combined.isna().mean() * 100).round(2)
})
missing_audit = missing_audit[missing_audit["missing_count"] > 0].sort_values(
    "missing_count", ascending=False
)
print("Missing-value audit (before handling):")
display(missing_audit)

# Count parsing failures separately from source nulls.
date_parse_issues = {
    "trending_date_missing_or_unparseable": int(combined["trending_date"].isna().sum()),
    "publish_time_missing_or_unparseable": int(combined["publish_time"].isna().sum()),
}
print("Date parsing issues:", date_parse_issues)

# Required-field rows are not suitable for video/date-based analysis.
before_required_drop = len(combined)
combined = combined.dropna(subset=["video_id", "country", "trending_date", "publish_time", "views"])
required_rows_dropped = before_required_drop - len(combined)

# Handle nullable engagement values and non-essential text.
combined["likes"] = combined["likes"].fillna(0)
combined["comment_count"] = combined["comment_count"].fillna(0)
combined["dislikes"] = combined["dislikes"].fillna(0)
combined["description"] = combined["description"].fillna("")

# Explicit duplicate definition: same video, same trending date, same country.
duplicate_key = ["video_id", "trending_date", "country"]
duplicates_removed = int(combined.duplicated(subset=duplicate_key).sum())
combined = combined.drop_duplicates(subset=duplicate_key, keep="first").copy()

# Exclude removed/errored videos from analysis; keep the count in the audit.
error_flag_count = int(combined["video_error_or_removed"].fillna(False).sum())
analysis_df = combined.loc[
    ~combined["video_error_or_removed"].fillna(False)
].copy()

# Demonstrate dropna() explicitly and report its effect (should be zero after required-field cleaning).
before_demo_dropna = len(analysis_df)
analysis_df = analysis_df.dropna(subset=["video_id", "trending_date", "publish_time", "views"])
dropna_rows_removed = before_demo_dropna - len(analysis_df)

print({
    "raw_rows": len(combined_raw),
    "rows_dropped_for_missing_required_fields": required_rows_dropped,
    "duplicate_rows_removed_by_key": duplicates_removed,
    "removed_or_errored_rows_excluded": error_flag_count,
    "additional_rows_removed_by_final_dropna": dropna_rows_removed,
    "analysis_rows": len(analysis_df),
})
print("Remaining missing values:")
display(analysis_df.isna().sum()[analysis_df.isna().sum() > 0])


Missing-value audit (before handling):


,missing_count,missing_percent
description,1624,3.51
comment_count,36,0.08
likes,12,0.03


Date parsing issues: {'trending_date_missing_or_unparseable': 0, 'publish_time_missing_or_unparseable': 0}
{'raw_rows': 46230, 'rows_dropped_for_missing_required_fields': 0, 'duplicate_rows_removed_by_key': 230, 'removed_or_errored_rows_excluded': 92, 'additional_rows_removed_by_final_dropna': 0, 'analysis_rows': 45908}
Remaining missing values:


Series([], dtype: int64)

### Data-quality decisions (summary)

- **Missing likes / comment_count:** fill with `0`; missing interaction counts should not be treated as positive engagement.
- **Missing description:** fill with `""`; descriptions are not used in numerical analysis.
- **Missing essential ID/date/views:** drop rows because they cannot be reliably assigned to a video, country, date, or view metric.
- **Duplicate:** same `video_id`, `trending_date`, and `country`; keep the first row.
- **Removed/errored video flag:** report and exclude from engagement analysis.
- **Date parsing:** parse `trending_date` with `%y.%d.%m` and `publish_time` as UTC; invalid values become `NaT` and are handled as missing essential dates.
- **Numeric fields:** convert with `pd.to_numeric(errors="coerce")` so invalid text becomes missing and can be audited.


## 3. NumPy arrays: shape, dtype, indexing and slicing

The four requested numeric columns are converted to NumPy arrays. Missing numeric values (if any remain) are represented as zero for array operations. We display shape and dtype for every array, the 10 largest view counts, every 100th record, and a small example sub-array.


In [4]:
array_source = analysis_df[["views", "likes", "dislikes", "comment_count"]].fillna(0)
views_arr = array_source["views"].to_numpy(dtype=np.float64)
likes_arr = array_source["likes"].to_numpy(dtype=np.float64)
dislikes_arr = array_source["dislikes"].to_numpy(dtype=np.float64)
comments_arr = array_source["comment_count"].to_numpy(dtype=np.float64)

for name, arr in {
    "views_arr": views_arr,
    "likes_arr": likes_arr,
    "dislikes_arr": dislikes_arr,
    "comments_arr": comments_arr,
}.items():
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

print("\\n10 largest view counts:")
print(np.sort(views_arr)[-10:][::-1])

print("\\nEvery 100th view record (first 20 shown):")
print(views_arr[::100][:20])

print("\\nExample sub-array (first 5 rows, 4 metrics):")
print(array_source.to_numpy(dtype=np.float64)[:5, :])


views_arr: shape=(45908,), dtype=float64
likes_arr: shape=(45908,), dtype=float64
dislikes_arr: shape=(45908,), dtype=float64
comments_arr: shape=(45908,), dtype=float64
\n10 largest view counts:
[3.60760164e+08 3.43297002e+08 2.86714357e+08 2.82812154e+08
 2.55362869e+08 2.36275528e+08 2.30704137e+08 2.10747294e+08
 2.00444300e+08 1.81785943e+08]
\nEvery 100th view record (first 20 shown):
[  48555.   21869.  298718.  642584.   11899.   66900.  840813.  457095.
 1073787.  212717.  798063.  138117. 1022589.  236091.  424662.   61550.
  481746. 1974327.  660529.   38994.]
\nExample sub-array (first 5 rows, 4 metrics):
[[4.85550e+04 2.18900e+03 1.80000e+02 1.36000e+02]
 [1.98385e+05 1.15900e+03 4.92000e+02 1.44600e+03]
 [2.31656e+05 8.74200e+03 8.96000e+02 4.32000e+02]
 [8.08300e+03 1.76000e+02 8.00000e+00 1.90000e+01]
 [3.73161e+05 1.13050e+04 1.67000e+02 8.91000e+02]]


## 4. Engagement score using NumPy only (no explicit loops)

- `like_ratio = likes / views` and `comment_rate = comment_count / views`; `np.divide` with `where` prevents division by zero.
- Min-max normalization maps views, likes and comments to the range 0–1. A constant array maps to zeros.
- **Chosen weights:** 50% normalized likes, 30% normalized views, 20% normalized comments. Likes receive the largest weight as a direct positive-interaction signal; views represent reach; comments contribute conversation but are weighted lower because comment counts are often smaller and can be noisy.
- `log1p(views)` compresses the long-tailed view distribution so very large values do not dominate a plot.
- A boolean mask flags records at or above the 90th percentile of engagement score (top 10% threshold).


In [5]:
def minmax_scale(values):
    values = np.asarray(values, dtype=np.float64)
    minimum = np.nanmin(values)
    maximum = np.nanmax(values)
    if maximum == minimum:
        return np.zeros_like(values, dtype=np.float64)
    return (values - minimum) / (maximum - minimum)

like_ratio = np.divide(
    likes_arr, views_arr,
    out=np.zeros_like(likes_arr, dtype=np.float64),
    where=views_arr != 0
)
comment_rate = np.divide(
    comments_arr, views_arr,
    out=np.zeros_like(comments_arr, dtype=np.float64),
    where=views_arr != 0
)

views_norm = minmax_scale(views_arr)
likes_norm = minmax_scale(likes_arr)
comments_norm = minmax_scale(comments_arr)

engagement_score = (
    0.50 * likes_norm +
    0.30 * views_norm +
    0.20 * comments_norm
)
log_views = np.log1p(views_arr)

engagement_cutoff = np.quantile(engagement_score, 0.90)
top_10_percent_mask = engagement_score >= engagement_cutoff

analysis_df = analysis_df.copy()
analysis_df["like_ratio"] = like_ratio
analysis_df["comment_rate"] = comment_rate
analysis_df["views_norm"] = views_norm
analysis_df["likes_norm"] = likes_norm
analysis_df["comments_norm"] = comments_norm
analysis_df["engagement_score"] = engagement_score
analysis_df["log_views"] = log_views
analysis_df["top_10_percent_engagement"] = top_10_percent_mask

print("Engagement weights: likes=0.50, views=0.30, comments=0.20")
print("90th percentile cutoff:", engagement_cutoff)
print("Records flagged by top-10% threshold:", int(top_10_percent_mask.sum()))
display(analysis_df[[
    "video_id", "country", "views", "likes", "comment_count",
    "like_ratio", "comment_rate", "engagement_score", "log_views",
    "top_10_percent_engagement"
]].head())
print("Note: raw views are strongly right-skewed; log1p compresses large values and makes the distribution easier to visualize.")


Engagement weights: likes=0.50, views=0.30, comments=0.20
90th percentile cutoff: 0.004820062852615465
Records flagged by top-10% threshold: 4591


,video_id,country,views,likes,comment_count,like_ratio,comment_rate,engagement_score,log_views,top_10_percent_engagement
0,g_N-w8LeAi8,US,48555,2189.0,136.0,0.045083,0.002801,0.000114,10.790473,False
1,Ugyql2NlPAd,US,198385,1159.0,1446.0,0.005842,0.007289,0.000557,12.197970,False
2,8fE19jkxJ9n,US,231656,8742.0,432.0,0.037737,0.001865,0.000461,12.353013,False
3,WOY0nn7HCsO,US,8083,176.0,19.0,0.021774,0.002351,0.000014,8.997642,False
4,2Lc_Cs5VVr_,US,373161,11305.0,891.0,0.030295,0.002388,0.000744,12.829768,False


Note: raw views are strongly right-skewed; log1p compresses large values and makes the distribution easier to visualize.


## 5. Reshape and broadcast: daily counts by country

We create a complete daily date index and a `days × countries` matrix of trending-record counts. `reshape` preserves the matrix shape and `flatten` returns a one-dimensional view. Broadcasting calculates (1) each country's share of each day's total and (2) each daily country count relative to that country's average daily count. No explicit Python loops are used in these array calculations.


In [6]:
daily_counts = (
    analysis_df.groupby(["trending_date", "country"])
    .size()
    .unstack("country", fill_value=0)
    .reindex(columns=["US", "GB", "IN"], fill_value=0)
)
all_days = pd.date_range(
    analysis_df["trending_date"].min(),
    analysis_df["trending_date"].max(),
    freq="D"
)
daily_counts = daily_counts.reindex(all_days, fill_value=0)
daily_matrix = daily_counts.to_numpy(dtype=np.float64)

print("daily_matrix shape:", daily_matrix.shape, "dtype:", daily_matrix.dtype)
reshaped_daily = daily_matrix.reshape(daily_matrix.shape[0], 3)
flattened_daily = reshaped_daily.flatten()
print("reshaped shape:", reshaped_daily.shape, "dtype:", reshaped_daily.dtype)
print("flattened shape:", flattened_daily.shape, "dtype:", flattened_daily.dtype)

daily_totals = daily_matrix.sum(axis=1, keepdims=True)
country_share = np.divide(
    daily_matrix, daily_totals,
    out=np.zeros_like(daily_matrix),
    where=daily_totals != 0
)
country_daily_average = daily_matrix.mean(axis=0, keepdims=True)
daily_vs_country_average = np.divide(
    daily_matrix, country_daily_average,
    out=np.zeros_like(daily_matrix),
    where=country_daily_average != 0
)

print("Country share matrix shape:", country_share.shape)
print("Daily value / country average matrix shape:", daily_vs_country_average.shape)
print("First 5 daily rows (US, GB, IN):")
print(daily_matrix[:5])


daily_matrix shape: (80, 3) dtype: float64
reshaped shape: (80, 3) dtype: float64
flattened shape: (240,) dtype: float64
Country share matrix shape: (80, 3)
Daily value / country average matrix shape: (80, 3)
First 5 daily rows (US, GB, IN):
[[199. 199. 199.]
 [200. 200. 200.]
 [199. 199. 200.]
 [200. 200. 199.]
 [200. 200. 200.]]


## 6. Filtering with boolean indexing, `loc`, `iloc` and sorting

The following cells answer all three filtering questions. Category-name filtering is completed after the category lookup merge below; here we first show the high-views/high-likes filter and the top 20 engagement records.


In [7]:
# (a) At least 1 million views AND more than 50,000 likes.
high_views_likes = analysis_df.loc[
    (analysis_df["views"] >= 1_000_000) & (analysis_df["likes"] > 50_000),
    ["video_id", "title", "channel_title", "country", "views", "likes"]
].sort_values(["views", "likes"], ascending=False)
print("Records with 1M+ views and >50K likes:", len(high_views_likes))
display(high_views_likes.head(30))

# (b) Top 20 records by engagement score.
top_20_engagement = analysis_df.sort_values(
    "engagement_score", ascending=False
).loc[:, [
    "video_id", "title", "channel_title", "country",
    "views", "likes", "comment_count", "engagement_score"
]].head(20)
display(top_20_engagement)

# Demonstrate iloc row/column positional selection.
print("iloc example — first 5 rows and first 6 columns:")
display(analysis_df.iloc[:5, :6])

# Demonstrate OR boolean logic using the two high-engagement signals.
high_views_or_likes = analysis_df.loc[
    (analysis_df["views"] >= 1_000_000) | (analysis_df["likes"] > 50_000),
    ["video_id", "country", "views", "likes", "engagement_score"]
].sort_values("engagement_score", ascending=False)
print("OR-filter result count:", len(high_views_or_likes))
display(high_views_or_likes.head(10))


Records with 1M+ views and >50K likes: 6705


,video_id,title,channel_title,country,views,likes
26526,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,360760164,27537822.0
30954,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,343297002,9581910.0
26325,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,286714357,21885701.0
30753,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,282812154,7893691.0
24108,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,255362869,11413793.0
26123,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,236275528,18035565.0
30552,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,230704137,6439282.0
25922,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,210747294,16086924.0
23907,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,200444300,8959133.0
30352,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,181785943,5073905.0


,video_id,title,channel_title,country,views,likes,comment_count,engagement_score
26526,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,360760164,27537822.0,598600.0,0.953805
26325,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,286714357,21885701.0,475737.0,0.758036
30954,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,343297002,9581910.0,663209.0,0.629861
26123,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,236275528,18035565.0,392045.0,0.624682
24108,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,255362869,11413793.0,778390.0,0.619592
25922,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,210747294,16086924.0,349687.0,0.557189
30753,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,282812154,7893691.0,546360.0,0.518887
23907,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,200444300,8959133.0,610988.0,0.486342
25721,RdqSDDng84n,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,169181540,12914095.0,280718.0,0.447294
30552,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,230704137,6439282.0,445693.0,0.423282


iloc example — first 5 rows and first 6 columns:


,video_id,trending_date,title,channel_title,category_id,publish_time
0,g_N-w8LeAi8,2017-11-14,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14 07:04:28+00:00
1,Ugyql2NlPAd,2017-11-14,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12 22:35:48+00:00
2,8fE19jkxJ9n,2017-11-14,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13 08:40:22+00:00
3,WOY0nn7HCsO,2017-11-14,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27 22:21:47+00:00
4,2Lc_Cs5VVr_,2017-11-14,Surprising Election Recipe,UltimateVlog96,27,2017-11-09 13:20:20+00:00


OR-filter result count: 10000


,video_id,country,views,likes,engagement_score
26526,RdqSDDng84n,GB,360760164,27537822.0,0.953805
26325,RdqSDDng84n,GB,286714357,21885701.0,0.758036
30954,QYwpwP6y0Eb,GB,343297002,9581910.0,0.629861
26123,RdqSDDng84n,GB,236275528,18035565.0,0.624682
24108,WQEWsMmimzZ,GB,255362869,11413793.0,0.619592
25922,RdqSDDng84n,GB,210747294,16086924.0,0.557189
30753,QYwpwP6y0Eb,GB,282812154,7893691.0,0.518887
23907,WQEWsMmimzZ,GB,200444300,8959133.0,0.486342
25721,RdqSDDng84n,GB,169181540,12914095.0,0.447294
30552,QYwpwP6y0Eb,GB,230704137,6439282.0,0.423282


## 7. Join category names and filter Music / Gaming

A **left merge** keeps every analysis record even if a category ID is absent from the lookup. We validate that the lookup has one row per `category_id` and compare row counts before and after the merge to demonstrate that no records were lost.


In [8]:
categories = pd.read_csv(CATEGORY_FILE)
categories["category_id"] = pd.to_numeric(categories["category_id"], errors="coerce")
categories = categories.dropna(subset=["category_id"]).drop_duplicates(subset=["category_id"])

rows_before_merge = len(analysis_df)
analysis_df = analysis_df.merge(
    categories[["category_id", "category_name"]],
    on="category_id",
    how="left",
    validate="many_to_one"
)
rows_after_merge = len(analysis_df)

print("Merge type: left")
print("Rows before merge:", rows_before_merge)
print("Rows after merge:", rows_after_merge)
print("Records lost:", rows_before_merge - rows_after_merge)
print("Unmatched category IDs:", int(analysis_df["category_name"].isna().sum()))
analysis_df["category_name"] = analysis_df["category_name"].fillna("Unknown")

overall_median_views = analysis_df["views"].median()
music_gaming_above_median = analysis_df.loc[
    analysis_df["category_name"].isin(["Music", "Gaming"]) &
    (analysis_df["views"] > overall_median_views),
    ["video_id", "title", "country", "category_name", "views", "likes", "engagement_score"]
].sort_values("views", ascending=False)
print("Overall median views:", overall_median_views)
print("Music/Gaming records above median:", len(music_gaming_above_median))
display(music_gaming_above_median.head(30))


Merge type: left
Rows before merge: 45908
Rows after merge: 45908
Records lost: 0
Unmatched category IDs: 0
Overall median views: 318570.0
Music/Gaming records above median: 9106


,video_id,title,country,category_name,views,likes,engagement_score
26350,RdqSDDng84n,Lena Ortiz - Monsoon (Live),GB,Music,360760164,27537822.0,0.953805
30738,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,GB,Music,343297002,9581910.0,0.629861
26151,RdqSDDng84n,Lena Ortiz - Monsoon (Live),GB,Music,286714357,21885701.0,0.758036
30539,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,GB,Music,282812154,7893691.0,0.518887
23954,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,GB,Music,255362869,11413793.0,0.619592
25951,RdqSDDng84n,Lena Ortiz - Monsoon (Live),GB,Music,236275528,18035565.0,0.624682
30339,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,GB,Music,230704137,6439282.0,0.423282
25751,RdqSDDng84n,Lena Ortiz - Monsoon (Live),GB,Music,210747294,16086924.0,0.557189
23755,WQEWsMmimzZ,Rook & Vale - Phones (Official Video) WATCH,GB,Music,200444300,8959133.0,0.486342
30139,QYwpwP6y0Eb,The Hollow Pines - Phones (Official Video) WATCH,GB,Music,181785943,5073905.0,0.333529


## 8. Aggregation: by category and by country

`groupby()` with `agg()` reports record count, average views, average likes and average engagement score. `value_counts()` is used for both category counts and percentages.


In [9]:
category_aggregation = (
    analysis_df.groupby("category_name")
    .agg(
        record_count=("video_id", "size"),
        average_views=("views", "mean"),
        average_likes=("likes", "mean"),
        average_engagement_score=("engagement_score", "mean")
    )
    .sort_values("record_count", ascending=False)
)
country_aggregation = (
    analysis_df.groupby("country")
    .agg(
        record_count=("video_id", "size"),
        average_views=("views", "mean"),
        average_likes=("likes", "mean"),
        average_engagement_score=("engagement_score", "mean")
    )
    .sort_values("record_count", ascending=False)
)
display(category_aggregation.round(2))
display(country_aggregation.round(2))

category_counts = analysis_df["category_name"].value_counts(dropna=False)
category_percentages = (
    analysis_df["category_name"].value_counts(normalize=True, dropna=False) * 100
).round(2)
category_distribution = pd.DataFrame({
    "count": category_counts,
    "percentage": category_percentages
})
display(category_distribution)


,record_count,average_views,average_likes,average_engagement_score
category_name,,,,
Entertainment,12244,901972.82,28962.50,0.00
Music,10286,3193066.33,135042.54,0.01
People & Blogs,3797,551729.44,19849.88,0.00
Comedy,3251,602812.39,28654.47,0.00
News & Politics,2420,283899.88,3112.20,0.00
Film & Animation,2290,578834.93,17218.50,0.00
Gaming,2055,509738.93,20752.44,0.00
Sports,1919,374671.12,10452.16,0.00
Howto & Style,1819,300403.17,10240.16,0.00


,record_count,average_views,average_likes,average_engagement_score
country,,,,
US,15968,1014564.29,39555.69,0.0
GB,14970,1583958.72,65270.77,0.0
IN,14970,953217.37,33178.28,0.0


,count,percentage
category_name,,
Entertainment,12244,26.67
Music,10286,22.41
People & Blogs,3797,8.27
Comedy,3251,7.08
News & Politics,2420,5.27
Film & Animation,2290,4.99
Gaming,2055,4.48
Sports,1919,4.18
Howto & Style,1819,3.96


### Publishing weekday and hour

The analysis compares average views by publishing weekday and UTC publishing hour. The maximum average-view weekday/hour is reported; ties are retained in the result table.


In [10]:
analysis_df["publish_weekday"] = analysis_df["publish_time"].dt.day_name()
analysis_df["publish_hour_utc"] = analysis_df["publish_time"].dt.hour

weekday_views = (
    analysis_df.groupby("publish_weekday")["views"]
    .agg(record_count="count", average_views="mean")
    .sort_values("average_views", ascending=False)
)
hour_views = (
    analysis_df.groupby("publish_hour_utc")["views"]
    .agg(record_count="count", average_views="mean")
    .sort_values("average_views", ascending=False)
)
print("Weekday ranking by average views:")
display(weekday_views.round(2))
print("Highest-average-view weekday(s):")
display(weekday_views[weekday_views["average_views"] == weekday_views["average_views"].max()])
print("UTC hour ranking by average views:")
display(hour_views.round(2))
print("Highest-average-view hour(s):")
display(hour_views[hour_views["average_views"] == hour_views["average_views"].max()])


Weekday ranking by average views:


,record_count,average_views
publish_weekday,,
Monday,6695,1592955.82
Friday,6256,1285997.38
Tuesday,7087,1257604.84
Thursday,6585,1111214.57
Wednesday,6534,1103036.83
Saturday,6475,973685.53
Sunday,6276,913033.79


Highest-average-view weekday(s):


,record_count,average_views
publish_weekday,,
Monday,6695,1.592956e+06


UTC hour ranking by average views:


,record_count,average_views
publish_hour_utc,,
12,1854,3251512.14
8,1849,2082956.46
16,2043,1519496.84
21,1843,1432900.76
15,1971,1352449.58
14,2029,1310911.45
7,1962,1242850.99
6,2023,1225602.16
10,2103,1194263.74


Highest-average-view hour(s):


,record_count,average_views
publish_hour_utc,,
12,1854,3.251512e+06


## 9. Time to trend and trending lifespan

`days_to_trend` is the calendar-day difference between `trending_date` and `publish_time`. Trending lifespan is the number of **distinct trending dates** for each `video_id` within each country. We list the ten longest-lasting video-country combinations and calculate average lifespan by category.


In [11]:
publish_date = analysis_df["publish_time"].dt.tz_convert(None).dt.normalize()
analysis_df["days_to_trend"] = (
    analysis_df["trending_date"].dt.normalize() - publish_date
).dt.days

print("days_to_trend distribution:")
display(analysis_df["days_to_trend"].describe().to_frame().T)
print("Negative days_to_trend rows (audit):", int((analysis_df["days_to_trend"] < 0).sum()))

video_lifespan = (
    analysis_df.groupby(["video_id", "country", "title", "channel_title", "category_name"])
    .agg(
        trending_lifespan_days=("trending_date", "nunique"),
        max_views=("views", "max")
    )
    .reset_index()
)
longest_lasting_videos = video_lifespan.sort_values(
    ["trending_lifespan_days", "max_views"],
    ascending=[False, False]
).head(10)
display(longest_lasting_videos)

average_lifespan_by_category = (
    video_lifespan.groupby("category_name")["trending_lifespan_days"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("average_lifespan_days")
)
display(average_lifespan_by_category.round(2))


days_to_trend distribution:


,count,mean,std,min,25%,50%,75%,max
days_to_trend,45908.0,5.090943,6.243572,0.0,1.0,3.0,6.0,47.0


Negative days_to_trend rows (audit): 0


,video_id,country,title,channel_title,category_name,trending_lifespan_days,max_views
6651,RdqSDDng84n,GB,Lena Ortiz - Monsoon (Live),FinalSpeech81,Music,25,360760164
5748,Nc1MUQvmyWO,US,Sunny Park - Cats (Official Video),ShockingInterview30,Music,25,97553795
5800,NqokCmZykzK,GB,Anya Lux - Paris (Lyric Video),QuietInterview9,Music,25,97513776
9679,dIbJkzpClRM,GB,Sunny Park - Robots (Live),SurprisingUpdate52,Music,25,60754408
9602,cz0dTlZWwSk,GB,Final Cars Speech WATCH,UltimateUnboxing65,Film & Animation,25,60717690
2987,BsTMdCTYs7P,US,Surprising Guitar Update,Brand NewUnboxing21,Entertainment,25,54982508
4377,Hh9b0BBsne0,GB,Kairo - Phones (Lyric Video)!,PerfectTrailer65,Music,25,43802674
3625,EZaG9_2Yi1u,GB,KAIRO - FESTIVAL (LIVE),QuietChallenge75,Music,25,38242672
3377,DYLzrqG-SrK,GB,Official Football Update,InsaneReaction46,Entertainment,25,26603358
5864,O6zdCRwWdf1,GB,MIRA SOLIS - WINTER (LYRIC VIDEO)?,FinalTrailer17,Music,25,17542788


,average_lifespan_days
category_name,
Music,5.32
Entertainment,3.53
Comedy,2.79
People & Blogs,2.78
Film & Animation,2.70
Gaming,2.64
Education,2.44
Howto & Style,2.40
Autos & Vehicles,2.37


## 10. Concatenation and channel-level summary merge

The country DataFrames are concatenated into one table. A per-video summary uses each video's maximum observed views within a country to avoid adding the same video's daily view snapshots together. The per-channel summary then counts distinct videos and sums those per-video maximum views. A merge attaches the channel-level totals to each video-summary row.


In [12]:
# Re-concatenate the cleaned country-specific data to demonstrate concat().
cleaned_country_frames = [
    analysis_df.loc[analysis_df["country"] == country].copy()
    for country in ["US", "GB", "IN"]
]
combined_clean = pd.concat(cleaned_country_frames, ignore_index=True)
print("Combined cleaned shape:", combined_clean.shape)
display(combined_clean.head())

per_video_summary = (
    combined_clean.groupby(["channel_title", "video_id", "country"])
    .agg(
        max_video_views=("views", "max"),
        trending_days=("trending_date", "nunique")
    )
    .reset_index()
)

per_channel_summary = (
    per_video_summary.groupby("channel_title")
    .agg(
        trending_video_count=("video_id", "nunique"),
        total_views=("max_video_views", "sum"),
        countries_present=("country", "nunique")
    )
    .reset_index()
)

video_with_channel_summary = per_video_summary.merge(
    per_channel_summary,
    on="channel_title",
    how="left",
    validate="many_to_one"
).sort_values(["total_views", "trending_video_count"], ascending=False)

print("Per-video summary:")
display(per_video_summary.head())
print("Per-channel summary:")
display(per_channel_summary.sort_values("total_views", ascending=False).head(20))
print("Merged video + channel summary:")
display(video_with_channel_summary.head(20))


Combined cleaned shape: (45908, 29)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio,comment_rate,views_norm,likes_norm,comments_norm,engagement_score,log_views,top_10_percent_engagement,category_name,publish_weekday,publish_hour_utc,days_to_trend
0,g_N-w8LeAi8,2017-11-14,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14 07:04:28+00:00,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,0.045083,0.002801,0.000132,0.000079,0.000175,0.000114,10.790473,False,People & Blogs,Tuesday,7,0
1,Ugyql2NlPAd,2017-11-14,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12 22:35:48+00:00,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,0.005842,0.007289,0.000547,0.000042,0.001858,0.000557,12.197970,False,News & Politics,Sunday,22,2
2,8fE19jkxJ9n,2017-11-14,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13 08:40:22+00:00,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,0.037737,0.001865,0.000639,0.000317,0.000555,0.000461,12.353013,False,Music,Monday,8,1
3,WOY0nn7HCsO,2017-11-14,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27 22:21:47+00:00,Festival|Guitar,8083,176.0,8,19.0,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,0.021774,0.002351,0.000019,0.000006,0.000024,0.000014,8.997642,False,Music,Friday,22,18
4,2Lc_Cs5VVr_,2017-11-14,Surprising Election Recipe,UltimateVlog96,27,2017-11-09 13:20:20+00:00,Festival,373161,11305.0,167,891.0,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,0.030295,0.002388,0.001031,0.000411,0.001145,0.000744,12.829768,False,Education,Thursday,13,5


Per-video summary:


,channel_title,video_id,country,max_video_views,trending_days
0,AmazingBreakdown51,Dtm-J99mwxO,US,252518,2
1,AmazingBreakdown51,_oEmb5yZjME,GB,19185,1
2,AmazingBreakdown51,rMhkXID6GEd,IN,77854,2
3,AmazingBreakdown51,xh-jdnGrWMw,IN,262668,1
4,AmazingChallenge30,NmXHnVxbq1Z,GB,266150,2


Per-channel summary:


,channel_title,trending_video_count,total_views,countries_present
290,QuietChallenge75,466,1339792039,3
432,WildUnboxing39,56,582623682,3
133,FinalTrailer17,161,504969156,3
281,PerfectTutorial78,98,480943573,3
278,PerfectTrailer65,66,453629190,3
132,FinalSpeech81,34,443010583,3
359,SurprisingHighlights41,123,403780627,3
248,OfficialReview41,457,398413392,3
294,QuietInterview9,88,342335691,3
31,Brand NewChallenge45,302,311324020,3


Merged video + channel summary:


,channel_title,video_id,country,max_video_views,trending_days,trending_video_count,total_views,countries_present
9428,QuietChallenge75,--pb5oKkYrH,GB,582515,4,466,1339792039,3
9429,QuietChallenge75,-9Kz9JXnpsj,GB,149149,1,466,1339792039,3
9430,QuietChallenge75,-Drx1CduAdT,GB,515878,1,466,1339792039,3
9431,QuietChallenge75,-GKkAA69O4n,IN,322767,2,466,1339792039,3
9432,QuietChallenge75,-RpvC1OMP5f,IN,595199,1,466,1339792039,3
9433,QuietChallenge75,-XqFhEmmdzn,US,6373327,13,466,1339792039,3
9434,QuietChallenge75,-YF8XpkQ-FV,US,206916,3,466,1339792039,3
9435,QuietChallenge75,-k24_cPwyMA,IN,2615269,2,466,1339792039,3
9436,QuietChallenge75,-v4YZ68EsrI,US,5533056,3,466,1339792039,3
9437,QuietChallenge75,-z-q1AiGLMv,US,551661,4,466,1339792039,3


## 11. Final checks and concise evidence summary

This final section prints key counts that can be quoted in the Part 2 README. Use the actual outputs from your run; do not replace them with estimates.


In [13]:
print("FINAL PART 2 CHECKS")
print("--------------------")
print("Raw rows:", len(combined_raw))
print("Countries:", sorted(combined_raw["country"].dropna().unique().tolist()))
print("Rows after key de-duplication and error exclusion:", len(analysis_df))
print("Duplicate key rows removed:", duplicates_removed)
print("Rows flagged removed/errored:", error_flag_count)
print("Missing values before handling:")
display(missing_audit)
print("Category merge records lost:", rows_before_merge - rows_after_merge)
print("NumPy array shapes:", {
    "views": views_arr.shape,
    "likes": likes_arr.shape,
    "dislikes": dislikes_arr.shape,
    "comments": comments_arr.shape,
})
print("Top 10% engagement cutoff:", engagement_cutoff)
print("Average days to trend:", round(analysis_df["days_to_trend"].mean(), 2))
print("Average trending lifespan:", round(video_lifespan["trending_lifespan_days"].mean(), 2))


FINAL PART 2 CHECKS
--------------------
Raw rows: 46230
Countries: ['GB', 'IN', 'US']
Rows after key de-duplication and error exclusion: 45908
Duplicate key rows removed: 230
Rows flagged removed/errored: 92
Missing values before handling:


,missing_count,missing_percent
description,1624,3.51
comment_count,36,0.08
likes,12,0.03


Category merge records lost: 0
NumPy array shapes: {'views': (45908,), 'likes': (45908,), 'dislikes': (45908,), 'comments': (45908,)}
Top 10% engagement cutoff: 0.004820062852615465
Average days to trend: 5.09
Average trending lifespan: 3.05


## Submission checklist

- [ ] This notebook runs top-to-bottom offline.
- [ ] `data/` contains the three country CSVs and `categories.csv`.
- [ ] NumPy shapes and dtypes are reported for all four arrays.
- [ ] Ratio, normalization, engagement score, log scaling and top-10% mask use vectorized NumPy operations.
- [ ] Daily country matrix, reshape, flatten and broadcasting are demonstrated.
- [ ] `head()`, `info()`, `describe()` and `shape` are shown for country and combined DataFrames.
- [ ] Missing values, duplicate definition/count, and all cleaning decisions are documented.
- [ ] `&`, `|`, `loc`, `iloc` and sorting are demonstrated.
- [ ] Left category merge retains every row.
- [ ] `groupby/agg`, `value_counts`, `concat` and `merge` are used.
- [ ] `days_to_trend`, video lifespan, longest-lasting videos and category lifespan are reported.
- [ ] Part 2 README and requirements file are committed inside `part2_numpy_pandas/`.
